## 1. Dataset Loading & Exploration

The objective of this project is to cluster patient testimonies using
unsupervised learning techniques.

The dataset contains textual descriptions of patients' symptoms.
No labels indicating the medical condition are provided.
The goal is to discover coherent patient groups automatically.

In [7]:
from Dataset import load_dataset

df = load_dataset("Student_Dataset.csv")

print(df.head())
print(df.info())
print(df.describe(include="all"))

   id     color                                               text
0   1       NaN  On uneven paths the heel feels wobbly side‑to‑...
1   2  0x000000  The feverish feeling eased, but I’m now breath...
2   3  0x000000  While working in the garden, I touched a metal...
3   4       NaN  A short sprint brought on a twinge in the midd...
4   5  0x000000  I accidentally left my hand on a hot mug for a...
<class 'pandas.DataFrame'>
RangeIndex: 1011 entries, 0 to 1010
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      1011 non-null   int64
 1   color   614 non-null    str  
 2   text    1011 non-null   str  
dtypes: int64(1), str(2)
memory usage: 174.1 KB
None
                 id  ...                                               text
count   1011.000000  ...                                               1011
unique          NaN  ...                                               1002
top             NaN  ...  My calf became red, w

In [8]:
print("Patients :", len(df))
print(df["color"].value_counts(dropna=False))

Patients : 1011
color
0x000000    614
NaN         397
Name: count, dtype: int64


## 2. Data Pre-processing

Since the dataset is composed of free-text medical testimonies,
the text must be transformed into a numerical representation before
it can be clustered.

The original CSV file is never modified.
All preprocessing is performed in memory during execution.

The preprocessing pipeline includes:

- lowercase conversion;
- English stop-word removal;
- filtering of extremely common words (`max_df`);
- filtering of very rare words (`min_df`);
- extraction of unigrams and bigrams;
- TF-IDF vectorization.

These operations reduce noise while preserving medically meaningful
expressions such as "lower back", "blood pressure" or "chest pain".

In [9]:
from Preprocessing import vectorize

X, vectorizer = vectorize(df["text"])

print(X.shape)

(1011, 3049)


## 3. Model Training

K-Means clustering is used to partition patient testimonies into groups
sharing similar symptom descriptions.

Several values of k were tested.
The silhouette score was used as an evaluation metric to compare
cluster quality.

In [11]:
from Clustering import compute_silhouette, create_clusters


scores = compute_silhouette(X)

model, labels = create_clusters(X, 7)

### Clusters in 2D

SVD reduces the texts to two dimensions. Each point is a text, and its color shows the cluster. This simplified view does not show all differences.

In [ ]:
import matplotlib.pyplot as plt
from Preprocessing import reduce_dimensions

points, svd = reduce_dimensions(X, components=2)

plt.figure(figsize=(8, 6))
for group in range(model.n_clusters):
    plt.scatter(points[labels == group, 0], points[labels == group, 1], label=group, s=15, alpha=0.7)
plt.legend(title='Cluster')
plt.xlabel('SVD 1')
plt.ylabel('SVD 2')
plt.title('Patient testimony clusters')
plt.tight_layout()
plt.show()

## 4. Loss Function

K-Means minimizes the Within-Cluster Sum of Squares (WCSS),
also called inertia.

Each patient is assigned to the nearest centroid.
The algorithm repeatedly updates the centroid positions until the
overall distance between patients and their assigned centroid is minimized.

Lower inertia corresponds to more compact clusters.

The silhouette score is then used as an external evaluation metric
to estimate how well-separated the resulting clusters are.

## 5. Model Comparison

This section compares the different preprocessing pipelines and
clustering configurations tested during the project.

| Pipeline        |  k | Silhouette |
| --------------- | -: | ---------: |
| TF-IDF + KMeans |  7 |     0.0122 |
| TF-IDF + KMeans | 10 |     0.0123 |
| TF-IDF + KMeans | 20 |     0.0124 |


## 6. Cluster Interpretation

To understand what each cluster represents,
the most representative keywords and several example testimonies
are displayed.

In [13]:
from Clustering import print_cluster_keywords

print_cluster_keywords(model, vectorizer)


Cluster 0
hot
feels
hand
skin
arm
blister
red
forearm
wrist
small
Cluster 1
makes
deep
ache
knee
sore
ankle
sharp
long
leg
make
Cluster 2
itchy
rash
new
spots
raised
hives
bumps
like
welts
appeared
Cluster 3
throat
nose
felt
mild
cough
runny
day
cold
congestion
didn
Cluster 4
pain
head
starts
feel
throbbing
throbbing pain
vision
pain starts
gets
like
Cluster 5
ear
pain
fever
hear
pain ear
left ear
cold
right ear
slight
right
Cluster 6
feel
chest
ve
feeling
breath
doctor
night
blood
pressure
notice


In [14]:
from Clustering import print_cluster_examples, get_cluster_names

cluster_names = get_cluster_names(model, vectorizer)
print_cluster_examples(df, labels, cluster_names)

Cluster 0
- While working in the garden, I touched a metal grill that had been on the stove. My left forearm has a burning itch and a faint scar.
- I accidentally left my hand on a hot mug for a minute. It now feels very tender, and there's a small blister.
- During a birthday party, a hot beverage was knocked onto my back. The area is red, tender, and a small blister is visible on my shoulder.
- On the sole, a painful spot feels like stepping on a stone. The surface is rough with tiny dark dots
- Where my collarbone meets the chest feels puffy by evening and aches when I push up from the floor. Reaching across my body brings a small jab there.
Cluster 1
- On uneven paths the heel feels wobbly side‑to‑side. Slanted pavements bring a quick aching shift under the ankle when I step.
- A short sprint brought on a twinge in the middle of the back of my thigh. It’s tender along a band and stiffens if I sit too long.
- A sudden twist while reaching for a bag made one side of my lower back fee

## 7. Conclusions & Insights

The initial clustering demonstrates that unsupervised learning is capable
of grouping patients with similar symptom descriptions without using
predefined labels.

Seven clusters were selected because they provide a good compromise
between interpretability and silhouette score.
Although higher values of k slightly increase the silhouette score,
they also produce clusters that are much more fragmented.

Future work will focus on improving the preprocessing pipeline
and comparing multiple clustering configurations.